# Running and Observing Agents

In [22]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "family": "llama",
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
    },
)


In [23]:
async def web_search(query: str) -> str:
    """Find information on the web"""
    return "The Labrador Retriever or simply Labrador is a British breed of retriever gun dog. "

In [24]:
agent = AssistantAgent(
    name='assistant',
    model_client=model_client,
    tools=[web_search],
    system_message=(
        "You are a research assistant. "
        "You MUST call the web_search tool for any information lookup. "
        "Do not answer from your own knowledge."
    ),
)

In [25]:
result = await agent.run(task='Find information about Labrador Retriever')
print(result.messages[-1].content)

The Labrador Retriever or simply Labrador is a British breed of retriever gun dog. 


### On_message() method 
 - we can see inside out agent
 - Inner messages produced by the agent, they can be BaseAgentEvent
 - A chat message produced by the agent as the response.

In [26]:
from autogen_core import CancellationToken

async def assistant_run()-> None:
    response = await agent.on_messages(
        messages= [TextMessage(content='Find information about Labrador Retriever',source='User')],
        cancellation_token=CancellationToken()
    )

    print(response.inner_messages)
    print('\n\n\n\n')
    print(response.chat_message)

await assistant_run()

[ToolCallRequestEvent(source='assistant', models_usage=RequestUsage(prompt_tokens=830, completion_tokens=32), metadata={}, content=[FunctionCall(id='ajs34a0tj', arguments='{"query":"Labrador Retriever"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='assistant', models_usage=None, metadata={}, content=[FunctionExecutionResult(content='The Labrador Retriever or simply Labrador is a British breed of retriever gun dog. ', name='web_search', call_id='ajs34a0tj', is_error=False)], type='ToolCallExecutionEvent')]





source='assistant' models_usage=None metadata={} content='The Labrador Retriever or simply Labrador is a British breed of retriever gun dog. ' type='ToolCallSummaryMessage'


In [27]:
from autogen_core import CancellationToken

async def assistant_run()-> None:
    response = await agent.on_messages(
        messages= [TextMessage(content='Find information about Labrador Retriever via the tool',source='User')],
        cancellation_token=CancellationToken()
    )

    print(response.inner_messages)
    print('\n\n\n\n')
    print(response.chat_message)

await assistant_run()

[ToolCallRequestEvent(source='assistant', models_usage=RequestUsage(prompt_tokens=914, completion_tokens=32), metadata={}, content=[FunctionCall(id='pagasbfm1', arguments='{"query":"Labrador Retriever"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(source='assistant', models_usage=None, metadata={}, content=[FunctionExecutionResult(content='The Labrador Retriever or simply Labrador is a British breed of retriever gun dog. ', name='web_search', call_id='pagasbfm1', is_error=False)], type='ToolCallExecutionEvent')]





source='assistant' models_usage=None metadata={} content='The Labrador Retriever or simply Labrador is a British breed of retriever gun dog. ' type='ToolCallSummaryMessage'


### Streaming Messages

#### on_messages_stream()
 - stream all the messages 

In [28]:
from autogen_agentchat.ui import Console


async def assistant_run_stream() -> None:

    await Console(
        agent.on_messages_stream(
        messages= [TextMessage(content='Find information about Labrador Retriever via the tool',source='User')],
        cancellation_token=CancellationToken()
    ),
    output_stats=True # Enable stats Printing
    )


await assistant_run_stream()

---------- ToolCallRequestEvent (assistant) ----------
[FunctionCall(id='jxmcvkga3', arguments='{"query":"Labrador Retriever"}', name='web_search')]
[Prompt tokens: 998, Completion tokens: 32]
---------- ToolCallExecutionEvent (assistant) ----------
[FunctionExecutionResult(content='The Labrador Retriever or simply Labrador is a British breed of retriever gun dog. ', name='web_search', call_id='jxmcvkga3', is_error=False)]
---------- assistant ----------
The Labrador Retriever or simply Labrador is a British breed of retriever gun dog. 
---------- Summary ----------
Number of inner messages: 2
Total prompt tokens: 998
Total completion tokens: 32
Duration: 0.48 seconds
